# Fall & Call: Emergency Keyword Spotting (KWS) v2.0
This notebook trains a CNN model to recognize **"HELP"**, **"EMERGENCY"**, **"CANCEL"** and **"BACKGROUND NOISE"**.

### 1. Setup and Hyperparameters

In [ ]:
import os
import numpy as np
import librosa
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix

# MFCC Parameters (Matching Lab 4 / Nano 33 Sense Specs)
N_MFCC = 13
SAMPLING_RATE = 16000
FRAME_SIZE = 512
HOP_LENGTH = 256
CLASSES = ['help', 'emergency', 'cancel', 'background']

### 2. Audio Processing (MFCC Extraction)
This function reads raw PDM text files, converts them to floats, and extracts MFCC features.

In [ ]:
def extract_mfcc(filename, class_id):
    print(f"Processing {filename}...")
    with open(filename, 'r') as f:
        lines = f.readlines()
    
    # Convert text to numeric samples
    raw_samples = [float(x.strip()) for x in lines if x.strip() and not x.startswith('-')]
    raw_samples = np.array(raw_samples)
    
    # Each gesture is 1 second (16000 samples)
    num_samples_per_word = 16000
    num_words = len(raw_samples) // num_samples_per_word
    
    mfccs = []
    for i in range(num_words):
        segment = raw_samples[i*num_samples_per_word : (i+1)*num_samples_per_word]
        
        # Extract MFCC
        mfcc = librosa.feature.mfcc(y=segment, sr=SAMPLING_RATE, 
                                    n_mfcc=N_MFCC, n_fft=FRAME_SIZE, 
                                    hop_length=HOP_LENGTH)
        # Transpose to (63, 13) shape
        mfccs.append(mfcc.T)
    
    labels = np.full((num_words,), class_id)
    return np.array(mfccs), labels

# Load data from collected files
try:
    X_h, y_h = extract_mfcc('help.txt', 0)
    X_e, y_e = extract_mfcc('emergency.txt', 1)
    X_c, y_c = extract_mfcc('cancel.txt', 2)
    X_b, y_b = extract_mfcc('background.txt', 3)

    X = np.vstack((X_h, X_e, X_c, X_b))
    y = np.concatenate((y_h, y_e, y_c, y_b))
    
    # Add channel dimension for CNN: (Samples, 63, 13, 1)
    X = X[..., np.newaxis]
    
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.1, random_state=42)
    
    print(f"\nSuccess! Total words processed: {len(X)}")
    print(f"Shape: {X.shape}")
except FileNotFoundError as e:
    print(f"Error: {e.filename} not found. Please collect audio data first.")

### 3. CNN Model Training
We use a lightweight CNN optimized for TinyML (Conv2D -> Depthwise -> Dense).

In [ ]:
model = models.Sequential([
    layers.Input(shape=(X.shape[1], X.shape[2], 1)),
    layers.Conv2D(16, (3, 3), activation='relu', padding='same'),
    layers.MaxPooling2D((2, 2)),
    layers.DepthwiseConv2D((3, 3), activation='relu', padding='same'),
    layers.MaxPooling2D((2, 2)),
    layers.Flatten(),
    layers.Dense(32, activation='relu'),
    layers.Dense(len(CLASSES), activation='softmax')
])

model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

history = model.fit(X_train, y_train, 
                    validation_data=(X_val, y_val), 
                    epochs=50, batch_size=4)

### 4. Evaluation and Export

In [ ]:
y_pred = np.argmax(model.predict(X_test), axis=1)
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', xticklabels=CLASSES, yticklabels=CLASSES, cmap='Purples')
plt.show()

# Export to TFLite
converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_model = converter.convert()
with open("emergency_model.tflite", "wb") as f:
    f.write(tflite_model)
print("Model saved as emergency_model.tflite")